In [6]:
%pip install tabulate pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 44.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 46.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

In [86]:
damage_per_rune_per_wave = 15
starting_mana = 115
passive_mana_per_wave = 20
rune_0_cost = (45 + 30 + 10) / 3.0
rune_inc_cost = 5 / 3.0

rune_costs = [rune_0_cost + i * rune_inc_cost for i in range(0, 100)]
rune_cum_cost = [sum(rune_costs[0:i + 1]) for i in range(0, 100)]

def find_last_index(arr, pred):
    return next((i for i in range(len(arr) - 1, -1, -1) if pred(arr[i])), -1)

def num_towers_for_mana_total(m):
    return 1 + find_last_index(rune_cum_cost, lambda c: c <= m)

def ratio_fn(total_hp, hpToNumRatio=0.7):
    numEnemies = round((total_hp / hpToNumRatio) ** 0.5) or 1
    enemyHp = round(total_hp / numEnemies) or 1
    return f"{numEnemies} @ {enemyHp}hp"

df = pd.DataFrame({'wave_index': range(0, 21)})
df['passive_mana_available'] = starting_mana + df['wave_index'] * passive_mana_per_wave
df['total_hp'] = 15 + 2.5 * df['wave_index'] ** 2;
df['approx_num'] = df['total_hp'].map(ratio_fn)
df['prev_wave_hp'] = df['total_hp'].shift(1).cumsum().fillna(0)
df['total_mana_available'] = df['passive_mana_available'] + df['prev_wave_hp']
df['num_runes'] = df['total_mana_available'].map(num_towers_for_mana_total)
df['damage_output'] = df['num_runes'] * damage_per_rune_per_wave

df['safety_factor'] = df['damage_output'] / df['total_hp']

df
# df[['total_hp', 'damage_output']]

,wave_index,passive_mana_available,total_hp,approx_num,prev_wave_hp,total_mana_available,num_runes,damage_output,safety_factor
0,0,115,15.0,5 @ 3hp,0.0,115.0,3,60,4.000000
1,1,135,17.5,5 @ 4hp,15.0,150.0,4,80,4.571429
2,2,155,25.0,6 @ 4hp,32.5,187.5,5,100,4.000000
3,3,175,37.5,7 @ 5hp,57.5,232.5,6,120,3.200000
4,4,195,55.0,9 @ 6hp,95.0,290.0,8,160,2.909091
5,5,215,77.5,11 @ 7hp,150.0,365.0,10,200,2.580645
6,6,235,105.0,12 @ 9hp,227.5,462.5,12,240,2.285714
7,7,255,137.5,14 @ 10hp,332.5,587.5,14,280,2.036364
8,8,275,175.0,16 @ 11hp,470.0,745.0,17,340,1.942857
9,9,295,217.5,18 @ 12hp,645.0,940.0,20,400,1.839080


In [63]:
df2 = df[['total_hp']]

for hpToNumRatio in [.1, .2, .4, .6, .7, .8, .9, 1.0, 1.1, 1.2, 1.3, 1.5, 1.7, 2.0, 3.0, 5.0, 10.0, 20.0, 40.0, 100]:
    df2[f"h/n: {hpToNumRatio}"] = df2['total_hp'].map(lambda h: ratio_fn(h, hpToNumRatio))

df2

,total_hp,h/n: 0.1,h/n: 0.2,h/n: 0.4,h/n: 0.6,h/n: 0.7,h/n: 0.8,h/n: 0.9,h/n: 1.0,h/n: 1.1,...,h/n: 1.3,h/n: 1.5,h/n: 1.7,h/n: 2.0,h/n: 3.0,h/n: 5.0,h/n: 10.0,h/n: 20.0,h/n: 40.0,h/n: 100
0,15.0,12 @ 1hp,9 @ 2hp,6 @ 2hp,5 @ 3hp,5 @ 3hp,4 @ 4hp,4 @ 4hp,4 @ 4hp,4 @ 4hp,...,3 @ 5hp,3 @ 5hp,3 @ 5hp,3 @ 5hp,2 @ 8hp,2 @ 8hp,1 @ 15hp,1 @ 15hp,1 @ 15hp,1 @ 15hp
1,17.5,13 @ 1hp,9 @ 2hp,7 @ 2hp,5 @ 4hp,5 @ 4hp,5 @ 4hp,4 @ 4hp,4 @ 4hp,4 @ 4hp,...,4 @ 4hp,3 @ 6hp,3 @ 6hp,3 @ 6hp,2 @ 9hp,2 @ 9hp,1 @ 18hp,1 @ 18hp,1 @ 18hp,1 @ 18hp
2,25.0,16 @ 2hp,11 @ 2hp,8 @ 3hp,6 @ 4hp,6 @ 4hp,6 @ 4hp,5 @ 5hp,5 @ 5hp,5 @ 5hp,...,4 @ 6hp,4 @ 6hp,4 @ 6hp,4 @ 6hp,3 @ 8hp,2 @ 12hp,2 @ 12hp,1 @ 25hp,1 @ 25hp,1 @ 25hp
3,37.5,19 @ 2hp,14 @ 3hp,10 @ 4hp,8 @ 5hp,7 @ 5hp,7 @ 5hp,6 @ 6hp,6 @ 6hp,6 @ 6hp,...,5 @ 8hp,5 @ 8hp,5 @ 8hp,4 @ 9hp,4 @ 9hp,3 @ 12hp,2 @ 19hp,1 @ 38hp,1 @ 38hp,1 @ 38hp
4,55.0,23 @ 2hp,17 @ 3hp,12 @ 5hp,10 @ 6hp,9 @ 6hp,8 @ 7hp,8 @ 7hp,7 @ 8hp,7 @ 8hp,...,7 @ 8hp,6 @ 9hp,6 @ 9hp,5 @ 11hp,4 @ 14hp,3 @ 18hp,2 @ 28hp,2 @ 28hp,1 @ 55hp,1 @ 55hp
5,77.5,28 @ 3hp,20 @ 4hp,14 @ 6hp,11 @ 7hp,11 @ 7hp,10 @ 8hp,9 @ 9hp,9 @ 9hp,8 @ 10hp,...,8 @ 10hp,7 @ 11hp,7 @ 11hp,6 @ 13hp,5 @ 16hp,4 @ 19hp,3 @ 26hp,2 @ 39hp,1 @ 78hp,1 @ 78hp
6,105.0,32 @ 3hp,23 @ 5hp,16 @ 7hp,13 @ 8hp,12 @ 9hp,11 @ 10hp,11 @ 10hp,10 @ 10hp,10 @ 10hp,...,9 @ 12hp,8 @ 13hp,8 @ 13hp,7 @ 15hp,6 @ 18hp,5 @ 21hp,3 @ 35hp,2 @ 52hp,2 @ 52hp,1 @ 105hp
7,137.5,37 @ 4hp,26 @ 5hp,19 @ 7hp,15 @ 9hp,14 @ 10hp,13 @ 11hp,12 @ 11hp,12 @ 11hp,11 @ 12hp,...,10 @ 14hp,10 @ 14hp,9 @ 15hp,8 @ 17hp,7 @ 20hp,5 @ 28hp,4 @ 34hp,3 @ 46hp,2 @ 69hp,1 @ 138hp
8,175.0,42 @ 4hp,30 @ 6hp,21 @ 8hp,17 @ 10hp,16 @ 11hp,15 @ 12hp,14 @ 12hp,13 @ 13hp,13 @ 13hp,...,12 @ 15hp,11 @ 16hp,10 @ 18hp,9 @ 19hp,8 @ 22hp,6 @ 29hp,4 @ 44hp,3 @ 58hp,2 @ 88hp,1 @ 175hp
9,217.5,47 @ 5hp,33 @ 7hp,23 @ 9hp,19 @ 11hp,18 @ 12hp,16 @ 14hp,16 @ 14hp,15 @ 14hp,14 @ 16hp,...,13 @ 17hp,12 @ 18hp,11 @ 20hp,10 @ 22hp,9 @ 24hp,7 @ 31hp,5 @ 44hp,3 @ 72hp,2 @ 109hp,1 @ 218hp
